# Eval Feedback Review
Compare human reviewer scores/feedback against automated eval scorers, identify discrepancies, and suggest prompt improvements.

## Setup

In [1]:
from legal_rag.eval_review import EvalReviewClient, ScorerResult
from config import *

import json 
import copy
import math
import os
import time
import pandas as pd

In [2]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [3]:
EVAL_RESULTS_FOLDER = os.path.join(EVAL_RESULTS_ROOT_PATH, f"eval_scorer_{EVAL_AI_PROMPT_VERSION}")
EVAL_RESULTS_FOLDER

'docs/KKP/LNC/eval_results/eval_scorer_v02'

## 1. Initialize Review Client

In [73]:
REVIEW_AI_PROMPT_VERSION = "v07"
NEW_EVAL_AI_PROMPT_VERSION = "v08"


reviewer = EvalReviewClient(
    eval_model=EVAL_AI_MODEL,
    review_model=REVIEW_AI_MODEL,
    eval_prompt_version=REVIEW_AI_PROMPT_VERSION,
    prompt_version=REVIEW_AI_PROMPT_VERSION,
)
reviewer

## 2. Load & Transform Eval Results JSON

In [74]:
data = EvalReviewClient.load_eval_folder(EVAL_RESULTS_FOLDER)

print(f"\nLoaded {len(data)} total entries\n")
for i, d in enumerate(data):
    print(f"Entry {i+1} [{d['source_file']}]: {d['input'][:80]}...")
    print(f"  Auto:  {d['auto_scores']}")
    print(f"  Human: {d['human_scores']}")
    print("-------------------------------")

  Loaded 5 entries from Deposit-Gemini-3.0 Flash Legal V02
  Loaded 5 entries from Deposit-Gemini-3.0 Pro Legal V02
  Loaded 15 entries from HP-Gemini-3.0 Flash Legal v02
  Loaded 20 entries from Lending-Gemini-3.0 Flash Legal V02

Total: 45 entries from 4 files

Loaded 45 total entries

Entry 1 [Deposit-Gemini-3.0 Flash Legal V02]: เจ้าของบัญชีเงินฝากถึงแก่กรรม บุคคลที่อ้างว่าเป็นทายาทขอดำเนินการจัดการบัญชี เช่...
  Auto:  {'reference': 0, 'judgement': 0, 'suggestion': 0}
  Human: {'reference': 0.5, 'judgement': 0.5, 'suggestion': 0}
-------------------------------
Entry 2 [Deposit-Gemini-3.0 Flash Legal V02]: นิติบุคคลเปิดบัญชีเงินฝาก ใช้หนังสือแสดงความประสงค์เปิดบัญชีแล้วลงนามโดยกรรมการผ...
  Auto:  {'reference': 0.5, 'judgement': 0, 'suggestion': 0}
  Human: {'reference': 0.5, 'judgement': 0.5, 'suggestion': 0.5}
-------------------------------
Entry 3 [Deposit-Gemini-3.0 Flash Legal V02]: เจ้าของบัญชีเงินฝากที่เสียชีวิตเป็นคนต่างชาติ ผู้ที่อ้างว่าเป็นทายาทมาขอถอนเงิน ...
  Auto:  

In [55]:
# Quick score comparison table (skip entries with None auto scores)
rows = []
for i, d in enumerate(data):
    for key in ["reference", "judgement", "suggestion"]:
        auto = d["auto_scores"][key]
        human = d["human_scores"][key]
        if human is None: continue
        rows.append({
            "entry": i + 1,
            "source_file": d["source_file"],
            "scorer": key,
            "auto_score": auto,
            "human_score": human,
            "diff": auto - human if auto is not None and human is not None else None,
        })

score_df = pd.DataFrame(rows)
score_df

,entry,source_file,scorer,auto_score,human_score,diff
0,1,Deposit-Gemini-3.0 Flash Legal V02,reference,0.0,0.5,-0.5
1,1,Deposit-Gemini-3.0 Flash Legal V02,judgement,0.0,0.5,-0.5
2,1,Deposit-Gemini-3.0 Flash Legal V02,suggestion,0.0,0.0,0.0
3,2,Deposit-Gemini-3.0 Flash Legal V02,reference,0.5,0.5,0.0
4,2,Deposit-Gemini-3.0 Flash Legal V02,judgement,0.0,0.5,-0.5
...,...,...,...,...,...,...
130,44,Lending-Gemini-3.0 Flash Legal V02,judgement,1.0,0.0,1.0
131,44,Lending-Gemini-3.0 Flash Legal V02,suggestion,1.0,0.5,0.5
132,45,Lending-Gemini-3.0 Flash Legal V02,reference,1.0,0.0,1.0
133,45,Lending-Gemini-3.0 Flash Legal V02,judgement,1.0,1.0,0.0


## 3. Scorer Review

Each scorer is reviewed independently through the same pipeline:
1. **Re-run** the scorer on all entries to get fresh scores + rationale
2. **Compare** re-run scores vs human scores
3. **Analyze discrepancies** between human feedback and auto rationale
4. **Suggest prompt improvements** for the next scorer version

In [56]:
SCORER_NAMES = ["reference", "judgement", "suggestion"]
LIMIT = 10
NUM_RERUN_ROWS = math.floor(LIMIT / len(SCORER_NAMES))
NUM_RERUN_TIMES = math.ceil(len(data) / NUM_RERUN_ROWS)


all_reviews = {}  # scorer_name -> list[EntryReview]
all_suggestions = {}  # scorer_name -> str

In [57]:
# new_reviews = []

# for i in reviews:
#     new_scorer_results = []
#     for j in i.scorer_results:
#         new_scorer_results.append(
#             ScorerResult(
#                 scorer_name=j['scorer_name'],
#                 score=j['score'],
#                 choice=j['choice'],
#                 rationale=j['rationale']
#             )
#         )
#     i.scorer_results = new_scorer_results
#     new_reviews.append(i)

# reviews = new_reviews
# reviews

In [58]:
reviews = []
# 3.1.1 Re-run scorer
for i in range(NUM_RERUN_TIMES):
    start_idx = i * NUM_RERUN_ROWS
    end_idx = min((i + 1) * NUM_RERUN_ROWS, len(data))
    print(f"\nRe-running scorers for entries {start_idx + 1} to {end_idx}...")
    reviews.extend(reviewer.rerun_scorers(data[start_idx:end_idx], SCORER_NAMES))

    print("Sleeping for 20 seconds to avoid rate limits...")
    time.sleep(20)  # Sleep to avoid hitting rate limits  
    print("------------------------------------------------")

reviews


Re-running scorers for entries 1 to 3...
  [1/3] reference: 0.0
  [1/3] judgement: 1.0
  [1/3] suggestion: 0.0
  [2/3] reference: 0.5
  [2/3] judgement: 0.0
  [2/3] suggestion: 0.0
  [3/3] reference: 0.5
  [3/3] judgement: 0.5
  [3/3] suggestion: 0.5
Sleeping for 20 seconds to avoid rate limits...
------------------------------------------------

Re-running scorers for entries 4 to 6...
  [1/3] reference: 0.5
  [1/3] judgement: 1.0
  [1/3] suggestion: 0.5
  [2/3] reference: 0.5
  [2/3] judgement: 1.0
  [2/3] suggestion: 1.0
  [3/3] reference: 0.0
  [3/3] judgement: 0.0
  [3/3] suggestion: 0.0
Sleeping for 20 seconds to avoid rate limits...
------------------------------------------------

Re-running scorers for entries 7 to 9...
  [1/3] reference: 0.0
  [1/3] judgement: 0.0
  [1/3] suggestion: 0.0
  [2/3] reference: 0.5
  [2/3] judgement: 0.5
  [2/3] suggestion: 0.5
  [3/3] reference: 0.5
  [3/3] judgement: 1.0
  [3/3] suggestion: 1.0
Sleeping for 20 seconds to avoid rate limits...
--

[EntryReview(index=0, input='เจ้าของบัญชีเงินฝากถึงแก่กรรม บุคคลที่อ้างว่าเป็นทายาทขอดำเนินการจัดการบัญชี เช่น ปิดบัญชีเงินฝากดังกล่าวโดยไม่ดำเนินการจัดตั้งผู้จัดการมรดกได้หรือไม่', output='ในฐานะผู้เชี่ยวชาญทางกฎหมายอาวุโสของธนาคาร ขอเรียนชี้แจงประเด็นข้อกฎหมายและแนวทางปฏิบัติกรณีเจ้าของบัญชีเงินฝากถึงแก่กรรม และทายาทขอจัดการบัญชีโดยไม่มีคำสั่งตั้งผู้จัดการมรดก ดังนี้\n\n## 1. มาตรากฎหมายที่เกี่ยวข้อง\n\nจากการตรวจสอบแหล่งข้อมูลกฎหมาย (File Store) พบมาตรากฎหมายที่เกี่ยวข้องโดยตรง ดังนี้:\n\n*   **ประมวลกฎหมายแพ่งและพาณิชย์ (กฎหมายทั่วไป)**\n    *   **มาตรา ๖๖๕** "ผู้รับฝากจำต้องคืนทรัพย์สินซึ่งรับฝากไว้นั้นให้แก่ผู้ฝาก หรือทรัพย์สินนั้นฝากในนามของผู้ใด ให้คืนแก่ผู้นั้น\n        แต่หากผู้ฝากทรัพย์ตาย ท่านให้คืนทรัพย์สินนั้นให้แก่ทายาท"\n    *   **มาตรา ๑๕๙๙** "เมื่อบุคคลใดตาย มรดกของบุคคลนั้นตกทอดแก่ทายาท\n        ทายาทอาจเสียไปซึ่งสิทธิในมรดกได้แต่โดยบทบัญญัติแห่งประมวลกฎหมายนี้หรือกฎหมายอื่น"\n    *   **มาตรา ๑๖๐๐** "ภายใต้บังคับของบทบัญญัติแห่งประมวลกฎหมายนี้ กองมรดกของผู้ตายได้แก่ท

In [59]:


with open(f'docs\\KKP\\LNC\\review_results_{REVIEW_AI_PROMPT_VERSION}.json', 'w', encoding='utf-8') as f:
    res_list = [] 
    for r in reviews:
        temp_dict = copy.deepcopy(r.__dict__)
        temp_res = []

        for score in temp_dict['scorer_results']:
            temp_res.append(score.__dict__)
            # temp_res.append(score)

        temp_dict['scorer_results'] = temp_res
        res_list.append(temp_dict)

    json.dump(res_list, f, indent=4, ensure_ascii=False)

In [60]:
# with open('docs\\KKP\\LNC\\review_results_v02.json', 'r', encoding='utf-8') as f:
#     reviews = json.loads(f.read())

# reviews

In [61]:
# 3.1.2 Compare re-run scores vs human scores
total_ref_df = {} 

for scorer_name in SCORER_NAMES:
    rerun_rows = []

    for review in reviews:
        sr = review.scorer_results[0]
        rerun_rows.append({
            "entry": review.index + 1,
            "source_file": review.source_file,
            "rerun_score": sr.score,
            "original_auto": review.auto_scores.get(scorer_name),
            "human_score": review.human_scores.get(scorer_name),
        })
        # sr = review['scorer_results'][0]
        # rerun_rows.append({
        #     "entry": review['index'] + 1,
        #     "source_file": review['source_file'],
        #     "rerun_score": sr['score'],
        #     "original_auto": review['auto_scores'].get(scorer_name),
        #     "human_score": review['human_scores'].get(scorer_name),
        # })

    ref_df = pd.DataFrame(rerun_rows)
    ref_df['dif'] = ref_df['rerun_score'] - ref_df['human_score']
    total_ref_df[scorer_name] = ref_df

total_ref_df

{'reference':     entry                         source_file  rerun_score  original_auto  \
 0       1  Deposit-Gemini-3.0 Flash Legal V02          0.0            0.0   
 1       2  Deposit-Gemini-3.0 Flash Legal V02          0.5            0.5   
 2       3  Deposit-Gemini-3.0 Flash Legal V02          0.5            0.5   
 3       1  Deposit-Gemini-3.0 Flash Legal V02          0.5            0.5   
 4       2  Deposit-Gemini-3.0 Flash Legal V02          0.5            1.0   
 5       3    Deposit-Gemini-3.0 Pro Legal V02          0.0            0.5   
 6       1    Deposit-Gemini-3.0 Pro Legal V02          0.0            0.5   
 7       2    Deposit-Gemini-3.0 Pro Legal V02          0.5            0.5   
 8       3    Deposit-Gemini-3.0 Pro Legal V02          0.5            0.5   
 9       1    Deposit-Gemini-3.0 Pro Legal V02          1.0            1.0   
 10      2       HP-Gemini-3.0 Flash Legal v02          1.0            NaN   
 11      3       HP-Gemini-3.0 Flash Legal v02     

In [62]:
for scorer_name in SCORER_NAMES:
    print(f"{scorer_name}: {total_ref_df[scorer_name]['dif'].abs().sum()}")

reference: 14.0
judgement: 13.0
suggestion: 12.5


In [63]:
total_ref_df['reference']

,entry,source_file,rerun_score,original_auto,human_score,dif
0,1,Deposit-Gemini-3.0 Flash Legal V02,0.0,0.0,0.5,-0.5
1,2,Deposit-Gemini-3.0 Flash Legal V02,0.5,0.5,0.5,0.0
2,3,Deposit-Gemini-3.0 Flash Legal V02,0.5,0.5,0.5,0.0
3,1,Deposit-Gemini-3.0 Flash Legal V02,0.5,0.5,0.0,0.5
4,2,Deposit-Gemini-3.0 Flash Legal V02,0.5,1.0,0.5,0.0
5,3,Deposit-Gemini-3.0 Pro Legal V02,0.0,0.5,0.5,-0.5
6,1,Deposit-Gemini-3.0 Pro Legal V02,0.0,0.5,0.5,-0.5
7,2,Deposit-Gemini-3.0 Pro Legal V02,0.5,0.5,0.0,0.5
8,3,Deposit-Gemini-3.0 Pro Legal V02,0.5,0.5,0.5,0.0
9,1,Deposit-Gemini-3.0 Pro Legal V02,1.0,1.0,0.5,0.5


In [ ]:
# for scorer_name in SCORER_NAMES:
    # print(scorer_name, '\n')
    # print(total_ref_df[scorer_name].groupby(['source_file']).agg({'dif': lambda x: x.abs().sum(), ''}))

SyntaxError: ':' expected after dictionary key (1630486363.py, line 3)

## Discrepancies Analysis

In [65]:
review_results = [] 

for review in reviews:
    review_results.append({
        "input": review.input,
        "output": review.output,
        "expected": review.expected, 
        "human_scores": review.human_scores,
        "auto_scores": review.auto_scores,
        "human_feedback": review.human_feedback,
        "scorer_rationale": "\n".join([result.rationale for result in review.scorer_results])}
    )

# review_results


In [66]:
dif_response = client.models.generate_content(model="gemini-3-pro-preview", 
                        contents=f"""
                        INSTRUCTION:
                        คุณคือ Agent ในการวิเคราะห์ feedback ที่มนุษย์ให้ไว้สำหรับคำตอบของโมเดลตอบคำถามทางกฎหมายของธนาคาร โดยประกอบด้วยการให้คะแนน 3 ส่วน
                        1. reference - การให้คะแนนความถูกต้องของกฎหมายอ้างอิง โดยจำเป็นต้องบอกกฎหมายให้ถูกต้องและครบถ้วนตามที่มนุษย์ให้ไว้ เกณฑ์การให้คะแนนดังนี้
                            1.1 "อ้างอิงมาตรากฎหมายถูกต้องและครอบคลุมตามเฉลย": 1.0
                            1.2 "อ้างอิงมาตรากฎหมายไม่ถูกต้อง หรือไม่เกี่ยวข้อง": 0.0
                            1.3 "อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น": 0.5

                        2. judgement - การให้คะแนนความสมเหตุสมผลของการวินิจฉัย โดยต้องสอดคล้องกับกฎหมายอ้างอิงที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน เกณฑ์การให้คะแนนดังนี้
                        
                            2.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม": 1.0
                            2.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            2.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
                        3. suggestion - การให้คะแนนความถูกต้องและความเหมาะสมของข้อสรุปและคำแนะนำ โดยต้องสอดคล้องกับกฎหมายอ้างอิงและการวินิจฉัยที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน และต้องเป็นประโยชน์ต่อธนาคาร เกณฑ์การให้คะแนนดังนี้

                            3.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง": 1.0
                            3.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            3.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5

                        จงสรุปข้อแตกต่างของ human_feedback และ scorer_rationale ของแต่ละคะแนน reference, judgement, suggestion 
                        รวมถึง human_score และ scorer_score จากข้อมูลอ้างอิงทั้งหมด ทุก records ว่ามีความแตกต่างกันอย่างไรบ้าง และมีจุดไหนที่ scorer_rationale ไม่สอดคล้องกับ human_feedback บ้าง
                        
                        IMPORTANT: 
                        - เน้นที่การให้คะแนนที่ไม่ตรงกับมนุษย์เป็นหลัก และวิเคราะห์ว่าทำไมโมเดลถึงให้คะแนนแบบนั้น โดยอิงจาก scorer_rationale ที่โมเดลให้มา เที่ยบกับ human_feedback ที่มนุษย์ให้มา
                        - สรุปเป็นภาพรวมโดยไม่ต้องลงรายละเอียดของแต่ละ record แต่ให้สรุปแนวโน้มและ pattern ที่พบจากข้อมูลทั้งหมด

                        โดยอิงจากข้อมูลนี้: {review_results}
                        โดยกำหนดให้:
                        - input คือคำถามทางกฎหมายที่ให้โมเดลตอบตามข้อมูลที่มีอยู่ใน File Store
                        - output คือคำตอบที่โมเดลให้มา
                        - expected คือคำตอบที่ถูกต้องหรือคำตอบที่มนุษย์คาดหวัง
                        - human_scores คือคะแนนที่มนุษย์ให้ไว้สำหรับคำตอบของโมเดลในแต่ละส่วน reference, judgement, suggestion
                        - auto_scores คือคะแนนที่โมเดลให้ไว้สำหรับคำตอบของโมเดลในแต่ละส่วน reference, judgement, suggestion
                        - human_feedback คือ feedback ที่มนุษย์ให้ไว้สำหรับคำตอบของโมเดล
                        - scorer_rationale คือเหตุผลที่โมเดลให้มาสำหรับคะแนนที่ได้
                            """,
                            
                config=types.GenerateContentConfig(
                    temperature=0
                ),)

In [67]:
print(dif_response.text)

สรุปข้อแตกต่างระหว่าง **Human Feedback** และ **Scorer Rationale** รวมถึงคะแนนที่ได้รับ จากการวิเคราะห์ข้อมูลทั้งหมด พบแนวโน้มและ Pattern ที่สำคัญดังนี้:

### ภาพรวม (Overview)
โดยภาพรวม **Scorer (AI)** มักจะให้คะแนน **สูงกว่า** Human (มนุษย์) ในทุกหมวด (Reference, Judgement, Suggestion) อย่างมีนัยสำคัญ โดย Scorer มักมองว่าคำตอบมีความถูกต้องและสมเหตุสมผลในตัวเอง (Self-consistent) หรือถูกต้องตามหลักทฤษฎีกฎหมายทั่วไป ในขณะที่ Human จะตรวจโดยยึด **"เฉลย (Expected)"** เป็นเกณฑ์ตายตัว และเน้นความถูกต้องในเชิงปฏิบัติงานธนาคาร (Banking Practice) ความแม่นยำของตัวอักษร (Verbatim) และความครบถ้วนของประเด็นที่กำหนดไว้

---

### 1. Reference (การอ้างอิงกฎหมาย)

**ข้อแตกต่างสำคัญ:**
*   **ความถูกต้องทุกตัวอักษร (Verbatim Check):**
    *   **Human:** เคร่งครัดมาก หากมีการตัดทอนเนื้อหา (ใช้เครื่องหมาย ...), การสรุปความเองโดยไม่ยกตัวบทมาทั้งหมด, การพิมพ์ผิด, การใช้ถ้อยคำกฎหมายเก่า (เช่น "สามีภริยา" แทน "คู่สมรส"), หรือการจัดรูปแบบย่อหน้าไม่ตรงกับต้นฉบับ (ไม่แยกวรรค) Human จะหักคะแนนทันที หรือให้ 0
    *

## Suggest Prompt Edit

In [ ]:
from skill import load_prompt

reference_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_reference_scorer", EVAL_AI_PROMPT_VERSION)
judge_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_judgement_scorer", EVAL_AI_PROMPT_VERSION)
suggestion_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_suggestion_scorer", EVAL_AI_PROMPT_VERSION)

In [69]:
suggest_change_response = client.models.generate_content(model="gemini-3-pro-preview", 
                        contents=f"""
                        INSTRUCTION:
                        คุณคือ Agent ในการแนะนำการปรับปรุง prompt จากความแตกต่างของ feedback ที่มนุษย์ให้ไว้สำหรับคำตอบของโมเดลตอบคำถามทางกฎหมายของธนาคาร โดยประกอบด้วย prompt ของ scorer การให้คะแนน 3 ส่วน
                        1. reference - การให้คะแนนความถูกต้องของกฎหมายอ้างอิง โดยจำเป็นต้องบอกกฎหมายให้ถูกต้องและครบถ้วนตามที่มนุษย์ให้ไว้ เกณฑ์การให้คะแนนดังนี้
                            1.1 "อ้างอิงมาตรากฎหมายถูกต้องและครอบคลุมตามเฉลย": 1.0
                            1.2 "อ้างอิงมาตรากฎหมายไม่ถูกต้อง หรือไม่เกี่ยวข้อง": 0.0
                            1.3 "อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น": 0.5

                        2. judgement - การให้คะแนนความสมเหตุสมผลของการวินิจฉัย โดยต้องสอดคล้องกับกฎหมายอ้างอิงที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน เกณฑ์การให้คะแนนดังนี้
                        
                            2.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม": 1.0
                            2.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            2.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
                        3. suggestion - การให้คะแนนความถูกต้องและความเหมาะสมของข้อสรุปและคำแนะนำ โดยต้องสอดคล้องกับกฎหมายอ้างอิงและการวินิจฉัยที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน และต้องเป็นประโยชน์ต่อธนาคาร เกณฑ์การให้คะแนนดังนี้

                            3.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง": 1.0
                            3.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            3.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
                            
                        จงสรุปข้อแตกต่างของ human_feedback และ scorer_rationale ของแต่ละคะแนน reference, judgement, suggestion ว่ามีความแตกต่างกันอย่างไรบ้าง และมีจุดไหนที่ scorer_rationale ไม่สอดคล้องกับ human_feedback บ้าง

                        โดยอิงจากข้อมูลนี้: 
                        - reference scorer prompt: {reference_prompt}
                        - judgement scorer prompt: {judge_prompt}
                        - suggestion scorer prompt: {suggestion_prompt}
                        - สรุปความแตกต่างที่พบ: {dif_response.text}

                        OUTPUT: JSON ที่มี key เป็น "reference", "judgement", "suggestion" และ value เป็นคำแนะนำในการปรับปรุง prompt สำหรับแต่ละ scorer โดยต้องระบุให้ชัดเจนว่าควรปรับปรุงตรงไหนของ prompt และปรับปรุงอย่างไรเพื่อให้ scorer ให้คะแนนที่สอดคล้องกับ human_feedback มากขึ้น
                        """,
                            
                config=types.GenerateContentConfig(
                    temperature=0
                ),)
print(suggest_change_response.text)

```json
{
  "reference": "ควรปรับปรุง Prompt ให้มีความเคร่งครัดเรื่อง 'ความถูกต้องระดับตัวอักษร (Verbatim)' และ 'รูปแบบ (Formatting)' มากขึ้น โดยเพิ่มคำสั่งดังนี้:\n1. เพิ่มเงื่อนไขให้ตรวจสอบการคัดลอกตัวบทกฎหมายว่าต้อง 'ครบถ้วนทุกตัวอักษร' ห้ามมีการตัดทอน (Truncation), ห้ามใช้เครื่องหมายละความ (...), และห้ามสรุปความเอง (Paraphrasing) หากพบให้หักคะแนนทันที\n2. ระบุให้ตรวจสอบความถูกต้องของถ้อยคำเฉพาะทาง (Wording) เช่น 'คู่สมรส' vs 'สามีภริยา' ต้องตรงตามฉบับกฎหมายที่อ้างอิง\n3. เพิ่มเกณฑ์การหักคะแนนหากมีการอ้างอิงมาตราที่ไม่เกี่ยวข้อง (Irrelevant/Noise) ปะปนเข้ามา หรือการจัดรูปแบบย่อหน้าไม่ตรงกับต้นฉบับ\n4. เปลี่ยนคำสั่งจาก 'ตรวจสอบว่าครอบคลุม' เป็น 'ตรวจสอบเทียบกับเฉลยแบบคำต่อคำ (Word-for-word comparison)'",
  "judgement": "ควรปรับปรุง Prompt ให้ลดน้ำหนักของ 'ความสมเหตุสมผลในตัวเอง (Self-consistency)' และเน้น 'ความถูกต้องตามธงของเฉลย (Adherence to Expected Key)' โดยเพิ่มคำสั่งดังนี้:\n1. ระบุชัดเจนว่า 'ห้ามให้คะแนนความสมเหตุสมผล หากข้อสรุป (Conclusion) หรือธงคำตอบขัดแย้งกับเฉลย' แม้ว่าคำ

## Suggest New Prompt

In [70]:
new_prompt_response = client.models.generate_content(model="gemini-3-pro-preview", 
                        contents=f"""
                        INSTRUCTION:
                        คุณคือ Agent ในการปรับปรุง prompt จากคำแนะนำ โดยประกอบด้วย prompt ของ scorer การให้คะแนน 3 ส่วน
                        1. reference - การให้คะแนนความถูกต้องของกฎหมายอ้างอิง โดยจำเป็นต้องบอกกฎหมายให้ถูกต้องและครบถ้วนตามที่มนุษย์ให้ไว้ เกณฑ์การให้คะแนนดังนี้
                            1.1 "อ้างอิงมาตรากฎหมายถูกต้องและครอบคลุมตามเฉลย": 1.0
                            1.2 "อ้างอิงมาตรากฎหมายไม่ถูกต้อง หรือไม่เกี่ยวข้อง": 0.0
                            1.3 "อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น": 0.5

                        2. judgement - การให้คะแนนความสมเหตุสมผลของการวินิจฉัย โดยต้องสอดคล้องกับกฎหมายอ้างอิงที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน เกณฑ์การให้คะแนนดังนี้
                        
                            2.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม": 1.0
                            2.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            2.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
                        3. suggestion - การให้คะแนนความถูกต้องและความเหมาะสมของข้อสรุปและคำแนะนำ โดยต้องสอดคล้องกับกฎหมายอ้างอิงและการวินิจฉัยที่ให้ไว้ และต้องมีเหตุผลสนับสนุนที่ชัดเจน และต้องเป็นประโยชน์ต่อธนาคาร เกณฑ์การให้คะแนนดังนี้

                            3.1 "ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง": 1.0
                            3.2 "ไม่ถูกต้องตามเฉลย": 0.0
                            3.3 "ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น": 0.5
                            
                        จงปรับปรุง prompt ของแต่ละ scorer โดยอิงจากคำแนะนำในการปรับปรุงที่ได้รับ 
                        
                        โดยอิงจากข้อมูลนี้: 
                        - current reference scorer prompt: {reference_prompt}
                        - current judgement scorer prompt: {judge_prompt}
                        - current suggestion scorer prompt: {suggestion_prompt}
                        - คำแนะนำในการปรับปรุง: {suggest_change_response.text}

                        OUTPUT: เพียง JSON เท่านั้น ที่มี key เป็น "reference", "judgement", "suggestion" และ value เป็น prompt ที่ถูกปรับปรุงใหม่สำหรับแต่ละ scorer
                        """,
                            
                config=types.GenerateContentConfig(
                    temperature=0
                ),)

In [71]:
# Extract JSON from the response text
json_text = new_prompt_response.text
start_idx = json_text.find('{')
end_idx = json_text.rfind('}') + 1
json_str = json_text[start_idx:end_idx]

# Parse and pretty print
new_prompts = json.loads(json_str)
print(json.dumps(new_prompts, indent=2, ensure_ascii=False))

{
  "reference": "คุณคือผู้เชี่ยวชาญทางกฎหมายที่ทำหน้าที่ประเมินคุณภาพคำตอบ และเลือกผลการประเมินตามตัวเลือกที่กำหนดให้\n\n# งานของคุณ\nให้คะแนนความถูกต้องของ **ข้อ 1 (มาตรากฎหมายที่เกี่ยวข้อง)** ในคำตอบ (OUTPUT) โดยเทียบกับเฉลย (EXPECTED) อย่างเคร่งครัดในระดับตัวอักษร (Verbatim)\n\n# รูปแบบข้อมูลเฉลย\nเฉลย (EXPECTED) อยู่ในรูปแบบ Markdown ที่แบ่งเป็นหัวข้อ:\n- `## 1. มาตรากฎหมายที่เกี่ยวข้อง` — มาตรากฎหมายและคำพิพากษาศาลฎีกาที่เกี่ยวข้อง (ใช้เป็นเฉลยหลักสำหรับการประเมินนี้)\n\n# เกณฑ์การประเมิน (Strict Verbatim & Formatting Check)\n1. **ความถูกต้องระดับตัวอักษร:** ตรวจสอบเทียบกับเฉลยแบบคำต่อคำ (Word-for-word comparison) ตัวบทกฎหมายต้อง **ครบถ้วนทุกตัวอักษร** \n   - ห้ามมีการตัดทอน (Truncation)\n   - ห้ามใช้เครื่องหมายละความ (...)\n   - ห้ามสรุปความเอง (Paraphrasing)\n   - ถ้อยคำเฉพาะทาง (Wording) เช่น 'คู่สมรส' vs 'สามีภริยา' ต้องตรงตามฉบับกฎหมายที่อ้างอิงในเฉลยเป๊ะๆ\n2. **ความถูกต้องของมาตรา:** เลขมาตราและชื่อกฎหมายต้องถูกต้องตรงกับเฉลย\n3. **สิ่งเจือปน (Noise):** ห้ามมีการอ้างอิงมาตร

In [ ]:
path = os.path.join("./skill_archive", "eval_scorer", EVAL_AI_MODEL, f"legal_reference_scorer_{NEW_EVAL_AI_PROMPT_VERSION}.md")
with open(path, "w", encoding="utf-8") as f:
    f.write(new_prompts["reference"])

    
path = os.path.join("./skill_archive", "eval_scorer", EVAL_AI_MODEL, f"legal_judgement_scorer_{NEW_EVAL_AI_PROMPT_VERSION}.md")
with open(path, "w", encoding="utf-8") as f:
    f.write(new_prompts["judgement"])

    
path = os.path.join("./skill_archive", "eval_scorer", EVAL_AI_MODEL, f"legal_suggestion_scorer_{NEW_EVAL_AI_PROMPT_VERSION}.md")
with open(path, "w", encoding="utf-8") as f:
    f.write(new_prompts["suggestion"])

## 4. Summary

All scorer reviews and prompt improvement suggestions are stored in `all_reviews` and `all_suggestions` for further use.